# 2. No negative or unanswerable questions
# 3. Threshold calibration
# 11. Full 512-token chunks are truncated during embedding
# 16. No comparison or ablation


### Student

Name: Vladimir Ungureanu <br>
Badge Number: 2141678 <br>
Master Program: Physics of Data <br>

## Domain: Applications of Machine Learning in Biomedicine

This project operates in the domain of **machine learning applied to biomedical
research**, using open-access articles from **PubMed Central (PMC)** published
within the last 12 months.

### Scope

The corpus covers the intersection of ML methods and biomedical problems,
including:

- **Clinical prediction** — diagnosis support, prognosis, disease risk scoring
- **Medical imaging** — segmentation, detection, and classification in
  radiology, pathology, and dermatology
- **Drug discovery & genomics** — molecule property prediction, variant
  calling, protein structure and function
- **Clinical NLP** — information extraction from EHRs, discharge summaries,
  and biomedical literature
- **ML methodology in biomedicine** — interpretability, federated learning,
  handling class imbalance and missing clinical data

### Corpus

| Property | Value |
|---|---|
| Source | PubMed Central (open access subset) |
| Query | `"machine learning"[Title/Abstract] AND open access[filter]` |
| Time window | Last 12 months |
| Selection | 500 full-text articles — the **most relevant** results from the top-1000 relevance-sorted PMC hits |
| Format | BioC XML → plain text → chunked |

### Why separate BioC XML markup from text and metadata?

The BioC API returns articles wrapped in XML markup (`<passage>`, `<text>`,
`<infon>`, `<offset>`...). The pipeline stores clean passage prose separately
from its structural metadata:

- **Embeddings do not receive XML tags.** The sentence-transformer model sees
  natural language prefixed only by the article title, semantic section label,
  and passage type.
- **Chunking uses clean input.** Token limits are measured over prose rather
  than XML syntax, while tokenizer character spans are retained for every
  chunk.
- **Structure is preserved.** Each article has a metadata sidecar containing
  its title, article identifiers, section types, passage types, original BioC
  offsets, and positions in the flattened text.
- **Citations remain traceable.** Chunk metadata is propagated through FAISS,
  retrieval, generation, and evaluation so answers can cite stable `[docN]`
  labels that map back to PMC articles and source offsets.

This design keeps the storage and modeling benefits of plain text without
losing article-level provenance.

Unzip the code package (`biomed_rag.zip`) and any data archives uploaded to Colab. Data archives are only extracted if their folder does not exist yet.

In [ ]:
import sys
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    content = Path("/content")
    for name in ["biomed_rag", "pmc_xml_test", "pmc_raw_texts", "document_repository", "vector_database"]:
        archive = content / f"{name}.zip"
        if not archive.exists() or (name != "biomed_rag" and (content / name).exists()):
            continue
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(content)
        print(f"Unzipped {archive.name}")

Install dependences not already installed in Google Colab. `bitsandbytes` is only needed when a model is loaded in 4-bit (`load_in_4bit=True`, off by default).

In [ ]:
%pip install -q faiss-cpu rank-bm25 accelerate "bitsandbytes>=0.46.1"

## Libraries

| Library | Purpose |
|---|---|
| `sentence-transformers` | Sentence embedding — encodes text chunks into dense vectors; cross-encoder reranking of retrieved candidates |
| `faiss` | Vector store — fast similarity search over chunk embeddings |
| `rank-bm25` | Lexical retrieval — BM25 scores fused with the dense results |
| `transformers` (`AutoModelForCausalLM`, `AutoTokenizer`) | Loads the pretrained LLM used for generation |
| `torch` | Deep learning backend — tensor ops and model inference (incl. quantization support) |


## Project structure

The pipeline code lives in the `biomed_rag/` package. This notebook runs it step by step and presents the results.

| Module | Responsibility |
|---|---|
| `config.py` | Paths, model IDs and PMC query settings |
| `utils.py` | JSON files and archives |
| `corpus.py` | PMC search, BioC XML download, section filtering, text extraction |
| `chunking.py` | Paragraph-aware chunking into ≤512-token segments |
| `stats.py` | Corpus and chunking statistics |
| `indexing.py` | Chunk embeddings and the FAISS index |
| `retrieval.py` | Hybrid retrieval: dense + BM25, fusion, reranking, merging |
| `llm.py` | Loading the language models, greedy generation, memory cleanup |
| `prompts.py` | All prompt templates |
| `testset.py` | Question and reference-answer generation |
| `rag.py` | RAG prompt and answering |
| `evaluation.py` | Retrieval metrics, LLM judge and the Markdown report |

**Running on Colab:** upload `biomed_rag.zip` (the zipped `biomed_rag/` folder) to `/content` together with the data archives (`pmc_xml_test.zip`, `pmc_raw_texts.zip`, and optionally `document_repository.zip` and `vector_database.zip`), then run all cells. After editing a module, upload a new `biomed_rag.zip` and re-run the first cell; `%autoreload` picks up the changes.

In [ ]:
%load_ext autoreload
%autoreload 2

from biomed_rag import chunking, config, corpus, evaluation, indexing, prompts, rag, retrieval, stats, testset
from biomed_rag.llm import LLM, free_memory
from biomed_rag.utils import make_zip, save_json

### IDs search

The search returns the top 1,000 most relevant PMC IDs for "machine learning"
published in the last year.

In [ ]:
query = corpus.build_pmc_query(days=365)
pmc_ids = corpus.search_pmc_ids(query, max_results=1000)
save_json(pmc_ids, config.PMC_IDS_PATH)
print("Found PMC IDs:", len(pmc_ids))
print(pmc_ids[:10])

### XLM download

Download the first 500 availabale XLMs of the found IDs

In [ ]:
failed_xml = corpus.download_corpus(pmc_ids, config.XML_DIR, n_articles=500)

### Find XLM unique sections



In [ ]:
if config.XML_DIR.exists():
    corpus.print_section_counts(corpus.count_section_types(config.XML_DIR))

## Excluding boilerplate sections

The BioC API labels every passage with `infon["section_type"]` (the section) and
`infon["type"]` (the passage role). After inspecting the labels used across a
200 article samples, they are filtered in 2 ways:

**By `section_type` — whole sections dropped:**

| `section_type` | What it contains | Passages (200-article sample) |
|---|---|---|
| `REF` | Bibliography / references | 12,622 |
| `ABBR` | Abbreviation lists | 1,359 |
| `SUPPL` | Supplementary material | 952 |
| `REVIEW_INFO` | Editorial metadata (received/accepted dates, editors) | 582 |
| `AUTH_CONT` | Author contributions | 532 |
| `COMP_INT` | Conflict of interest statements | 509 |
| `ACK_FUND` | Funding / acknowledgements | 149 |
| `APPENDIX` | Appendix material | 88 |

**By `type` - only the raw markup is dropped:**

| `type` | What it contains |
|---|---|
| `table` | Embedded `<table>` XML markup and algorithm pseudocode |

Table **captions** (`table_caption`) and **footnotes** (`table_footnote`) are
kept as they are short, readable prose that can help retrieval, while the raw
`<table>` XML itself is removed because even though it contains structured data,
it is not easily readable by LLMs. Figure captions (`FIG`) are kept for the same
reason as the table footnotes.

**Kept sections:** `TITLE`, `ABSTRACT`, `INTRO`, `METHODS`, `RESULTS`,
`DISCUSS`, `CONCL`, `FIG`(figure descriptions are informative), `TABLE` (captions/footnotes only).

In [ ]:
corpus.filter_sections(
    config.XML_DIR,
    config.FILTERED_XML_DIR,
    excluded_sections=config.EXCLUDED_SECTIONS,
    excluded_types=config.EXCLUDED_TYPES,
)

In [ ]:
corpus.extract_corpus_text(config.FILTERED_XML_DIR, config.RAW_TEXT_DIR)

## Paragraph-aware chunking into ≤512-token segments

Each article is chunked on its own, so a chunk never mixes two articles:

1. **Tokenization**: the article is tokenized with the embedding model's tokenizer, keeping the character offsets of every token.
2. **Paragraph-aware windows**: a chunk holds at most **512 tokens** and ends at the last paragraph boundary inside that window. Boundaries that would leave a chunk shorter than 128 tokens are ignored; if no boundary qualifies, the window is cut at a word boundary.
3. **Overlap**: consecutive chunks overlap by up to **64 tokens**, starting on a word boundary.
4. **Short tails**: if the last chunk of an article would be shorter than 128 tokens, it is replaced by the final 512 tokens of the article.
5. **Original text**: every chunk is a slice of the original text, taken through the token offsets rather than decoded from token IDs, so casing, numbers and symbols are preserved.

Chunks are saved as `<PMC_ID>_chunk_<NNNN>.txt`, numbered per article.

We tokenize with the same tokenizer as the embedding model used later - [`NeuML/pubmedbert-base-embeddings`](https://huggingface.co/NeuML/pubmedbert-base-embeddings). This guarantees that "512 tokens" is measured in the exact units the embedding model will consume (its `max_seq_length` is 512). Because that limit also counts the `[CLS]` and `[SEP]` tokens, a full 512-token chunk loses its last two tokens when it is embedded; except at the very end of an article, those tokens are also covered by the overlap with the next chunk.

In [ ]:
chunking.chunk_corpus(
    config.RAW_TEXT_DIR,
    config.CHUNK_DIR,
    tokenizer_id=config.EMBEDDING_MODEL_ID,
    chunk_tokens=512,
    overlap_tokens=64,
    min_chunk_tokens=128,
)

### Corpus and Chunking Statistics

The final corpus contains **500 biomedical articles**, divided into **10,526 text chunks** for embedding and retrieval. This corresponds to approximately **21.05 stored chunks per document**. The numbers below were computed locally on the same 500 articles; the next cell recomputes them. Tokens are counted without the `[CLS]` and `[SEP]` special tokens.

#### Chunk-level statistics

| Metric | Mean | Median | Standard deviation | Minimum | Maximum |
|---|---:|---:|---:|---:|---:|
| Characters per chunk | 2,288.6 | 2,351 | 500.5 | 391 | 3,848 |
| Words per chunk | 315.9 | 327 | 67.1 | 45 | 459 |
| Tokens per chunk | 422.6 | 443 | 80.0 | 132 | 512 |

#### Document-level statistics

| Metric | Mean | Median | Standard deviation | Minimum | Maximum |
|---|---:|---:|---:|---:|---:|
| Chunks per document | 21.1 | 19.5 | 9.3 | 1 | 75 |
| Characters per document | 40,977.6 | 37,624 | 18,450.7 | 1,490 | 144,465 |
| Words per document | 5,661.0 | 5,210.5 | 2,512.9 | 206 | 19,395 |
| Tokens per document | 7,583.8 | 6,939 | 3,307.0 | 239 | 25,291 |

In [ ]:
corpus_stats = stats.corpus_statistics(config.CHUNK_DIR, config.RAW_TEXT_DIR, tokenizer_id=config.EMBEDDING_MODEL_ID)
stats.print_corpus_report(corpus_stats)
save_json(corpus_stats, config.PROFILE_PATH)
print(f"\nSaved statistics to {config.PROFILE_PATH.name}")

### Choice of Embedding Model: `NeuML/pubmedbert-base-embeddings`

This project uses [`NeuML/pubmedbert-base-embeddings`](https://huggingface.co/NeuML/pubmedbert-base-embeddings) to represent document chunks and user queries as dense vectors. The model was selected because it is specifically designed for semantic similarity and retrieval in biomedical literature, which closely matches this project's corpus of PubMed Central articles about machine learning in biomedicine.

The model is based on **PubMedBERT**, whose pretraining data consists of biomedical abstracts and full-text articles. It was subsequently fine-tuned with the `sentence-transformers` framework using PubMed title–abstract pairs and related title pairs. Consequently, its vector representations are better adapted to biomedical terminology, abbreviations, and relationships than those produced by general-purpose embedding models.

The main reasons for selecting this model are:

- **Domain relevance:** its biomedical training data is closely aligned with the project's PMC corpus.
- **Semantic retrieval:** it is explicitly fine-tuned to map semantically related sentences and paragraphs to nearby points in the embedding space.
- **Biomedical vocabulary:** it can represent specialized clinical and scientific terminology more effectively than a general-purpose model.
- **Suitable output size:** each text is mapped to a **768-dimensional vector**, providing detailed semantic representations while remaining practical for FAISS indexing.
- **Long input support:** its maximum sequence length of **512 tokens** is appropriate for the document chunks used by the retrieval pipeline.
- **Retrieval performance:** according to the model card, it outperforms several general-purpose and biomedical alternatives on the reported PubMed semantic-similarity evaluations.

In [ ]:
indexing.build_index(config.CHUNK_DIR, config.VECTOR_DB_DIR, model_id=config.EMBEDDING_MODEL_ID, batch_size=128)

### Hybrid retrieval

Retrieval combines two rankings of the chunks:

- **Dense:** cosine similarity between PubMedBERT embeddings, searched with FAISS.
- **Lexical:** BM25 scores, which catch exact terms such as metric names, gene names and acronyms that embeddings can miss.

The two rankings are merged with reciprocal rank fusion (RRF), and a cross-encoder (`BAAI/bge-reranker-base`) reranks the top candidates. Duplicate chunks are dropped, a chunk adjacent to an already selected chunk of the same article is merged into it, and each hit is extended with its neighbouring chunks. If the best dense similarity is below `min_dense`, no passages are returned and the generator is told that no documents were retrieved.

### Choice of Reranking Model: `BAAI/bge-reranker-base`

Dense retrieval and BM25 first identify a relatively small set of potentially relevant chunks. The reranker then assigns a more precise relevance score to each question–passage pair and reorders the candidates before the final passages are given to the generative model.

Unlike the embedding model, which encodes the question and each passage independently, `bge-reranker-base` is a **cross-encoder**. It processes the question and passage together, allowing attention between their tokens. This makes it better able to recognize whether a passage directly answers the question, rather than merely discussing a semantically related topic. The additional precision is useful in biomedical retrieval, where several chunks may contain similar terminology while only one contains the requested metric, method, gene, result, or comparison.

The main reasons for selecting this model are:

- **Purpose-built reranking:** it is specifically designed to rerank a small candidate set returned by a faster first-stage retriever.
- **More precise relevance estimation:** joint question–passage processing can capture exact relationships that independent embeddings may miss.
- **Suitable for hybrid retrieval:** it can refine candidates found through both semantic similarity and exact-term BM25 matching.
- **Good efficiency–quality balance:** the model has approximately **278 million parameters** and is based on **XLM-RoBERTa-Base**, making it substantially lighter than `bge-reranker-large` while still providing cross-encoder reranking.
- **Practical for Google Colab:** only the top fused candidates are reranked, rather than the entire corpus, which keeps the additional computational cost manageable.
- **No domain-specific retraining required:** although it is not trained specifically for biomedicine, it can compare English questions and passages directly, while the biomedical PubMedBERT model remains responsible for domain-specific first-stage retrieval.

A cross-encoder is more computationally expensive than dense retrieval because it must process every question–passage pair separately. For this reason, it is used only on the top candidates rather than all document chunks. This two-stage design combines the speed of FAISS and BM25 with the greater precision of cross-encoder reranking.

### Retrieval configuration

| Parameter | Value | Motivation |
| --- | --- | --- |
| k_final | 5 | Returns at most five passage groups to the generator. This provides several sources of evidence without adding excessive context. |
| k_dense | 30 | Retrieves 30 semantic candidates with PubMedBERT and FAISS. A relatively broad candidate set improves recall when the correct chunk is relevant but not ranked first. |
| k_bm25 | 30 | Retrieves 30 lexical candidates. BM25 complements dense retrieval by matching exact biomedical terminology, acronyms, gene names, model names, and metrics such as RMSE or CV. |
| k_rerank | 30 | Sends only the 30 best fused candidates to the cross-encoder. This preserves a broad candidate pool while limiting the more expensive reranking step. |
| rrf_k | 60 | Controls reciprocal rank fusion. The conventional value of 60 reduces sensitivity to small rank differences and rewards chunks retrieved by both dense search and BM25. |
| min_dense | 0.45 | Refuses retrieval when the best dense cosine similarity is below 0.45. This reduces the risk of answering from unrelated passages.|
| expand_neighbours | True | Adds the chunks immediately before and after a selected chunk when they belong to the same article. This recovers evidence split across chunk boundaries. |
| use_reranker | True | Enables cross-encoder reranking, which compares each question and passage jointly and can distinguish directly answering passages from merely related ones. |

In [ ]:
retrieval_config = retrieval.RetrievalConfig(
    k_final=5,
    k_dense=30,
    k_bm25=30,
    k_rerank=30,
    rrf_k=60,
    min_dense=0.45,
    expand_neighbours=True,
    use_reranker=True,
)
retriever = retrieval.HybridRetriever.from_disk(config.VECTOR_DB_DIR, config.CHUNK_DIR, retrieval_config)

for hit in retriever.search("machine learning for disease diagnosis", k=5):
    print(f"{hit['score']:.3f}  {hit['chunk_file']}")
    print(hit["text"][:200], "\n")

### Choice of Generative Model: `LiquidAI/LFM2-1.2B-RAG`

This project uses [`LiquidAI/LFM2-1.2B-RAG`](https://huggingface.co/LiquidAI/LFM2-1.2B-RAG) as the generative component of the retrieval-augmented generation pipeline. The model receives the document chunks retrieved from the FAISS index and generates an answer grounded in that context.

The model was selected primarily because it is specifically fine-tuned for **retrieval-augmented generation**. Unlike a general-purpose language model, it is designed to answer questions from supplied documents.

The main reasons for selecting this model are:

- **RAG specialization:** the model is trained to identify and use relevant information from documents included in its prompt.
- **Grounded question answering:** conditioning generation on retrieved evidence helps reduce unsupported answers compared with relying only on the model's internal knowledge.
- **Compact model size:** with approximately **1.2 billion parameters**, it is substantially smaller than many general-purpose LLMs and is therefore suitable for environments with limited computational resources, such as Google Colab.

In this pipeline, the embedding model first converts the user's question into a vector. FAISS retrieves the most semantically relevant biomedical text chunks, and these chunks are inserted into the prompt expected by `LFM2-1.2B-RAG`. The model then produces an answer using the retrieved passages as evidence. This division of responsibilities combines the domain-specific retrieval capabilities of `NeuML/pubmedbert-base-embeddings` with a lightweight model specialized for context-grounded generation.

In [ ]:
print(prompts.RAG_SYSTEM_TEMPLATE)

with rag.RAGPipeline.load(retriever, model_id=config.GENERATOR_MODEL_ID) as generator:
    result = generator.answer("How is machine learning used for disease diagnosis?", k=retrieval_config.k_final, max_new_tokens=256)
print("\n" + result["answer"])

### Choice of Q/A Generation Model: `Qwen/Qwen3-4B-Instruct-2507`

This project uses [`Qwen/Qwen3-4B-Instruct-2507`](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) to generate questions and answers from the collected biomedical documents. The model receives a passage from a PubMed Central article and is instructed to produce Q/A pairs whose answers can be supported by the source text.

The model was selected because it provides a strong balance between **generation quality, instruction-following ability, context capacity, and computational cost**. With approximately **4 billion parameters**, it is more capable than very small language models while remaining practical to run in a constrained environment.

The main reasons for choosing this model are:

- **Strong instruction following:** the model is post-trained to follow natural-language instructions, which is important for controlling the required Q/A format.
- **Good text comprehension:** generating valid Q/A pairs requires identifying the main concepts, methods, results, and relationships described in a passage.
- **Improved scientific reasoning:** the model card reports improvements in science, logical reasoning, and knowledge-oriented tasks, which are relevant when processing biomedical literature.

### Prompt Engineering for Question Generation

The question-generation prompt was designed to create realistic evaluation queries from passages sampled from the biomedical document repository. The objective is not simply to reformulate a sentence as a question, but to simulate how a student would query the RAG system without having seen the source article.

The prompt is divided into four clearly labelled sections: **background**, **task**, **constraints**, and **output format**. This structure separates the model's role from the generation requirements and makes the instructions easier to follow.

#### Background and role definition

The model is instructed to behave as a student seeking information from a RAG system. Importantly, the simulated student is described as having no knowledge of the underlying article, dataset, analysis, or database.

This framing prevents questions such as:

> What method was used in this study?

Although such a question may be answerable from the passage, it is not a realistic standalone query because the user has not identified which study is being discussed. Instead, the prompt encourages self-contained questions such as:

> What methods are commonly used to select features in high-dimensional biomedical data?

The role definition therefore makes the generated questions better representations of queries that a real user could submit to the system.

#### Explicit generation task

The prompt asks for exactly **one clear and direct question** based on the supplied PubMed passage. Generating one question at a time reduces formatting errors and avoids producing several closely related questions from the same chunk.

The instruction to frame the question as if the model had never seen the source text is particularly important. Without it, language models frequently produce questions containing artificial references such as *“according to the passage”* or *“in this study.”* These references reveal that the question was generated from a document and would make the evaluation less representative of real RAG usage.

#### Constraints on question content

Several negative constraints are included to control common failure modes:

- The question must concern a meaningful **tool, metric, gene, method, or biomedical concept** found in the passage.
- It must be phrased as a broad, standalone student query.
- It must not refer to a *study*, *paper*, *author*, *dataset*, *analysis*, or *passage*.
- It must not contain expressions such as *“according to the text”* or *“in this paper.”*
- Its answer must be present in the supplied passage.
- The model must not provide an answer, explanation, or commentary.

These constraints serve two purposes. First, they make each question understandable without access to its source. Second, they ensure that the source passage can be treated as the reference evidence when evaluating retrieval and answer generation.

Requiring answerability from the passage is essential. A question may be topically related to a chunk while still requiring external knowledge. Such a question would not provide a reliable test of whether the RAG system retrieved and used the correct evidence.

#### Structured output

The model is required to return the question inside XML-style tags and it was used to to make the generated questions consistent, automatically extractable, and easy to validate.

```text
<question>Your generated question</question>

In [ ]:
print(prompts.QUESTION_GENERATION_TEMPLATE)

qa_llm = LLM.load(config.QA_MODEL_ID)
questions = testset.generate_questions(qa_llm, config.CHUNK_DIR, n_passages=20, seed=42)

### Prompt Engineering for Reference-Answer Generation

The answer-generation prompt is designed to produce a concise **reference answer** for each synthetic question. The answer must be grounded exclusively in the corresponding PubMed passage so that it can later serve as a reference when evaluating the RAG system.

The prompt is divided into four sections: **background**, **task**, **constraints**, and **output format**. This organization makes the role, source of evidence, expected behaviour, and output structure explicit.

#### Role definition

The prompt assigns the model the role of a precise biomedical question-answering assistant. This encourages the model to focus on factual information and use terminology appropriate for the biomedical domain.

The background also states that the assistant has no contextual knowledge of the article, study, dataset, or analysis from which the passage was taken. This discourages references to unspecified external material and helps produce an answer that can stand on its own.

#### Passage-grounded answering

The central instruction is:

> Answer the question using ONLY the provided passage.

This is the most important grounding constraint. Although the language model may possess relevant biomedical knowledge in its parameters, the reference answer must contain only information supported by the selected passage. Otherwise, the generated answer could include correct but externally derived facts that the RAG system cannot recover from the indexed corpus.

Using the passage as the sole source of evidence creates a controlled relationship between:

1. the generated question;
2. the source chunk;
3. the reference answer.

This relationship is required for meaningful retrieval and answer-quality evaluation.

#### Constraints

The constraints address common unwanted generation behaviours:

- **Use only directly stated facts:** the model should avoid unsupported inferences and information recalled from its internal knowledge.
- **Avoid source-related preambles:** expressions such as *“Based on the passage”* and *“According to the text”* are prohibited because the a user would not know what chunk the system is referring to.
- **Do not add commentary:** the model should return the answer itself rather than explanations about how it interpreted the question or followed the instructions.

Together, these restrictions aim to produce answers that are factual, direct, and suitable for comparison with the output of the RAG system.

#### Concise answer requirement

The output is explicitly described as a **concise answer**. Concision is useful because the generated output is intended to serve as a reference rather than a detailed article summary. A short answer:

- focuses on the information needed to answer the question;
- reduces the chance of introducing unsupported claims;
- facilitates comparison with the RAG-generated answer;
- avoids irrelevant background information.

The model is not necessarily required to copy a sentence verbatim. It may summarize or paraphrase the relevant information, provided that every fact remains supported by the passage.

#### Structured output

The answer must be enclosed in XML-style tags:

```text
<answer>Your concise answer</answer>

In [ ]:
print(prompts.GOLD_ANSWER_TEMPLATE)
 
test_set = testset.generate_gold_answers(qa_llm, questions)
qa_llm.close()
save_json(test_set, config.TEST_SET_PATH)
 
print(f"Test set: {len(test_set)} questions in {config.TEST_SET_PATH.name}")

In [ ]:
test_set = testset.load_test_set(
    config.TEST_SET_PATH,
    config.CHUNK_DIR,
)

## RAG answering on the test set

Each test question goes through the full pipeline: hybrid retrieval (`k=3`) followed by `LFM2-1.2B-RAG`. The answers and the retrieved passages are saved to `lfm_rag_answers.json`.

In [ ]:
with rag.RAGPipeline.load(
    retriever,
    model_id=config.GENERATOR_MODEL_ID,
) as generator:
    answers = rag.answer_test_set(
        generator,
        test_set,
        k=retrieval_config.k_final,
        max_new_tokens=100,
    )
 
save_json(answers, config.RAG_ANSWERS_PATH)


In [ ]:
answers = rag.load_answers(
    config.RAG_ANSWERS_PATH,
    test_set,
)

## Evaluation

- **Document retrieval:** *document hit@k* measures whether the gold article appears among the retrieved passage groups, while *document match@1* measures whether the first retrieved passage group comes from the gold article.
- **Chunk retrieval:** *exact chunk match@1* is a strict metric requiring the primary chunk of the first retrieved passage group to be the gold chunk. *Context chunk hit* measures whether the gold chunk appears anywhere in the final context supplied to the generative model, including primary retrieved chunks, merged adjacent chunks, and chunks added through neighbour expansion.
  
- **Answer evaluation:** `Qwen/Qwen3-4B-Instruct-2507` acts as an LLM judge. It receives the question, the complete retrieved context shown to the RAG model, the gold answer, and the RAG answer. It assigns two scores from 0 to 5:
  - **Correctness:** how accurately and completely the RAG answer addresses the question.
  - **Faithfulness:** whether every factual claim in the RAG answer is supported by the retrieved context.
- **Evaluation failures:** questions with an unusable gold answer are excluded from the judge averages. Judge outputs that cannot be parsed are counted separately instead of being assigned a score of zero.

The structured results are saved to `lfm_rag_answers_evaluated.json`. A human-readable version containing the summary metrics, individual answers, judge scores, and retrieved passages is saved to `evaluation_report.md`.

In [ ]:
print(prompts.JUDGE_TEMPLATE)

with LLM.load(config.QA_MODEL_ID) as judge:
    evaluated, summary = evaluation.evaluate(
        answers,
        judge,
    )

save_json(
    {"summary": summary, "records": evaluated},
    config.EVALUATION_PATH,
)
evaluation.write_markdown_report(
    evaluated,
    summary,
    config.REPORT_PATH,
)
evaluation.print_summary(summary)

In [ ]:
evaluated, summary = evaluation.load_evaluation(
    config.EVALUATION_PATH,
    answers,
)
evaluation.print_summary(summary)

In [ ]:
retriever.close()
free_memory(report=True)

### Export artifacts

Set `EXPORT_ARCHIVES = True` to zip `document_repository/` and `vector_database/`, so they can be downloaded and reused in a later session.

In [ ]:
EXPORT_ARCHIVES = False

if EXPORT_ARCHIVES:
    for directory in [config.CHUNK_DIR, config.VECTOR_DB_DIR]:
        print("Wrote", make_zip(directory))